<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 03 - Decision Tree: experimentos de hiperparámetros

En este notebook se realizan experimentos sencillos de ajuste de hiperparámetros para **Decision Tree**.

El objetivo es analizar cómo afecta la profundidad máxima del árbol (`max_depth`) al rendimiento del modelo y seleccionar la configuración con mejor valor de `f1`.

## 1. Importación de librerías y configuración inicial

Se importan las funciones de carga, entrenamiento, evaluación y guardado de modelos.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, save_metrics, save_model
from src.model_training import get_base_models, split_data

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Entrenamiento con distintas profundidades

Se prueban varios valores de `max_depth` usando `class_weight='balanced'` para compensar posibles desequilibrios entre clases.

In [ ]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
base = get_base_models(random_state=42)["decision_tree"]
experiments = []
for depth in [3, 5, 10, None]:
    candidate = base.__class__(random_state=42, class_weight="balanced", max_depth=depth)
    candidate.fit(X_train, y_train)
    experiments.append({
        "model": "decision_tree",
        "max_depth": depth,
        **evaluate_model(y_test, candidate.predict(X_test))
    })
results_df = pd.DataFrame(experiments)
best_depth = results_df.sort_values("f1", ascending=False).iloc[0]["max_depth"]
best_depth = None if pd.isna(best_depth) else int(best_depth)
best_model = base.__class__(random_state=42, class_weight="balanced", max_depth=best_depth)
best_model.fit(X_train, y_train)
results_df.sort_values("f1", ascending=False)

## 3. Guardado de resultados y mejor modelo

Se guardan las métricas de los experimentos y el mejor modelo entrenado.

In [ ]:
save_metrics(results_df.to_dict(orient="records"), RESULTS_DIR / "dt_grid_metrics.csv")
save_model(best_model, MODELS_DIR / "decision_tree_grid.joblib")

## 4. Conclusiones

Estos experimentos permiten controlar el sobreajuste del árbol de decisión. Una profundidad demasiado alta puede mejorar el entrenamiento, pero no necesariamente generalizar mejor.